# ARIA Pipeline (Colab Notebook)

This notebook runs the pipeline end-to-end in Google Colab. Run cells top to bottom.

## 1) Setup dependencies

In [ ]:
!pip -q install python-dotenv litellm tenacity datasets jailbreakbench pandas transformers accelerate bitsandbytes huggingface_hub


## 2) Get code into Colab
Option A (recommended): clone your repo.

In [ ]:
!git clone https://github.com/geryfabrega/ARIA.git
%cd /content/ARIA
!git checkout colab-edition
!git pull

If you uploaded a zip instead, unpack it and `cd` into the project root where `pipeline_colab/` exists.

## 3) API keys and Hugging Face setup


In [ ]:
import os
from getpass import getpass
from huggingface_hub import snapshot_download

os.environ["HF_TOKEN"] = getpass("HF_TOKEN (for model download): ")
os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY (for judge + feedback): ")

# Local models for attack workflow
ATTACKER_MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
TARGET_MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
os.environ["ATTACKER_MODEL_ID"] = ATTACKER_MODEL_ID
os.environ["TARGET_MODEL_ID"] = TARGET_MODEL_ID

print(f"Downloading attacker model: {ATTACKER_MODEL_ID} ...")
snapshot_download(repo_id=ATTACKER_MODEL_ID, token=os.environ["HF_TOKEN"])
print(f"Downloading target model: {TARGET_MODEL_ID} ...")
snapshot_download(repo_id=TARGET_MODEL_ID, token=os.environ["HF_TOKEN"])
print("Model downloads complete.")


## 4) Run attack pipeline (local HF model + OpenAI judge)
This now also re-tests each behavior's **final attack prompt** multiple times to compute final ASR.

In [ ]:
import os
import sys

# Ensure imports like `from attack_pipeline...` resolve from pipeline_colab/
sys.path.insert(0, "/content/ARIA/pipeline_colab")

from attack_pipeline import config as attack_config
from pipeline_colab.colab_workflows import run_attack_workflow

# Route attacker + target to different downloaded local HF models
attack_config.ATTACKER_MODEL = f"hf_local:{os.environ['ATTACKER_MODEL_ID']}"
attack_config.TARGET_MODEL = f"hf_local:{os.environ['TARGET_MODEL_ID']}"

final_prompts_csv = "outputs/final_attack_prompts_colab.csv"
final_asr_csv = "outputs/final_prompt_asr_colab.csv"

attack_csv = run_attack_workflow(
    openai_api_key=os.environ["OPENAI_API_KEY"],
    model_api_key="",
    behaviors=10,
    max_cycles=3,
    final_eval_attempts=10,
    output="outputs/attack_results_colab.csv",
    final_prompts_output=final_prompts_csv,
    final_asr_output=final_asr_csv,
)

(attack_csv, final_prompts_csv, final_asr_csv)


## 5) Preview tables: attack cycles and final ASR


In [ ]:
import pandas as pd

attack_df = pd.read_csv("outputs/attack_results_colab.csv")
cycle_table = (
    attack_df.groupby("cycle", as_index=False)
    .agg(attempts=("jailbroken", "size"), successes=("jailbroken", "sum"))
)
cycle_table["asr"] = (cycle_table["successes"] / cycle_table["attempts"]).round(4)
cycle_table


In [ ]:
import pandas as pd

final_prompts_df = pd.read_csv("outputs/final_attack_prompts_colab.csv")
final_asr_attempts_df = pd.read_csv("outputs/final_prompt_asr_colab.csv")

final_asr_table = final_prompts_df[[
    "behavior",
    "final_eval_attempts",
    "final_eval_successes",
    "final_asr",
]].sort_values("final_asr", ascending=False)

final_asr_table


## 6) Run judge comparison (optional)

In [ ]:
from pipeline_colab.colab_workflows import run_judge_comparison_workflow

details_csv, summary_csv = run_judge_comparison_workflow(
    together_api_key=os.environ["TOGETHERAI_API_KEY"],
    openai_api_key=os.environ["OPENAI_API_KEY"],
    samples=30,
    output="outputs/judge_comparison_results_colab.csv",
)

(details_csv, summary_csv)

## 7) Download outputs


In [ ]:
from google.colab import files

files.download("outputs/attack_results_colab.csv")
files.download("outputs/final_attack_prompts_colab.csv")
files.download("outputs/final_prompt_asr_colab.csv")
# files.download("outputs/judge_comparison_results_colab.csv")
# files.download("outputs/judge_comparison_results_colab_summary.csv")


## Optional: CLI-style execution
Use this if you prefer script commands over Python function calls.

In [ ]:
!python pipeline_colab/run_attack.py --behaviors 5 --max-cycles 5 --final-eval-attempts 10 --output outputs/my_run.csv --final-prompts-output outputs/my_final_prompts.csv --final-asr-output outputs/my_final_asr_attempts.csv
!python pipeline_colab/run_judge_comparison.py --samples 30 --output outputs/judge_comparison_results.csv